## 130 – Test de Repositoris i Unit of Work

Aquest notebook valida **tots** els mètodes obligatoris de cada repositori:
`add`, `get`, `get_all`, `update`, `delete`, `get_paginated` i les operacions de domini.

### Bloc 0 – Configuració del camí i imports

In [ ]:
import sys
sys.path.append('../src')

from domain import (
    engine, Session,
    SqlAlchemyUnitOfWork,
    User, FinancialProfile, Budget, BudgetCategory,
    Category, PaymentMethod, TransactionType,
    Transaction, Tag
)
from datetime import date
from decimal import Decimal

# Fàbrica de sessions per a la Unit of Work
from sqlalchemy.orm import sessionmaker
SessionFactory = sessionmaker(bind=engine)

print(f"Entorn: {engine.url}")

---
### Bloc 1 – `CategoryRepository`: add, get_all, get, update, delete

In [ ]:
# --- add() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    cat = Category(name="Test Alimentació", description="Per proves", is_essential=True)
    uow.categories.add(cat)
    uow.commit()
    print(f"[add] Categoria creada: id={cat.id_category}, nom='{cat.name}'")

In [ ]:
# --- get_all() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    cats = uow.categories.get_all()
    print(f"[get_all] Total categories: {len(cats)}")
    for c in cats:
        print(f"  id={c.id_category}  nom='{c.name}'  essencial={c.is_essential}")

In [ ]:
# --- get() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    cats = uow.categories.get_all()
    first_id = cats[0].id_category
    cat = uow.categories.get(first_id)
    print(f"[get] id={cat.id_category}  nom='{cat.name}'")

In [ ]:
# --- update() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    cats = uow.categories.get_all()
    cat = uow.categories.get(cats[-1].id_category)
    cat.description = "Descripció actualitzada"
    uow.categories.update(cat)
    uow.commit()
    print(f"[update] Categoria id={cat.id_category} actualitzada: desc='{cat.description}'")

In [ ]:
# --- get_essential_categories() (consulta específica) ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    essentials = uow.categories.get_essential_categories()
    print(f"[get_essential_categories] Total essencials: {len(essentials)}")
    for c in essentials:
        print(f"  id={c.id_category}  nom='{c.name}'")

---
### Bloc 2 – `PaymentMethodRepository` i `TransactionTypeRepository`

In [ ]:
# PaymentMethod: add + get_all
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    method = PaymentMethod(name="Targeta Visa", provider="Banc Sabadell")
    uow.payment_methods.add(method)
    uow.commit()
    all_methods = uow.payment_methods.get_all()
    print(f"[PaymentMethod] Total: {len(all_methods)}")
    for m in all_methods:
        print(f"  id={m.id_method}  nom='{m.name}'  proveïdor='{m.provider}'")

In [ ]:
# TransactionType: add + get_all
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    t1 = TransactionType(name="Ingrés")
    t2 = TransactionType(name="Despesa")
    uow.transaction_types.add(t1)
    uow.transaction_types.add(t2)
    uow.commit()
    types = uow.transaction_types.get_all()
    print(f"[TransactionType] Total: {len(types)}")
    for t in types:
        print(f"  id={t.id_type}  nom='{t.name}'")

---
### Bloc 3 – `UserRepository`: add, get, get_by_email, update, delete

In [ ]:
# --- add() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    user = User(
        name="Joan",
        surname="Prova",
        email="joan.prova@walletly.com",
        password="hashed_pass_123"
    )
    uow.users.add(user)
    uow.commit()
    print(f"[add] Usuari creat: id={user.id_user}  email='{user.email}'")

In [ ]:
# --- get() + get_by_email() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    users = uow.users.get_all()
    uid = users[0].id_user

    user_by_id = uow.users.get(uid)
    print(f"[get] id={user_by_id.id_user}  nom='{user_by_id.name}'")

    user_by_email = uow.users.get_by_email("joan.prova@walletly.com")
    if user_by_email:
        print(f"[get_by_email] Trobat: id={user_by_email.id_user}  email='{user_by_email.email}'")
    else:
        print("[get_by_email] No trobat")

In [ ]:
# --- update() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    users = uow.users.get_all()
    user = uow.users.get(users[0].id_user)
    user.name = "Joan Actualitzat"
    uow.users.update(user)
    uow.commit()
    print(f"[update] Usuari id={user.id_user} actualitzat: nom='{user.name}'")

---
### Bloc 4 – `BudgetRepository` i `add_budget_to_user` (operació de domini)

In [ ]:
# --- BudgetRepository: add + get_all ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    users = uow.users.get_all()
    user_id = users[0].id_user

    budget = Budget(
        id_user=user_id,
        month=5,
        year=2026,
        description="Pressupost Maig 2026",
        total_limit=Decimal("1500.00")
    )
    uow.budgets.add(budget)
    uow.commit()
    print(f"[add] Budget creat: id={budget.id_budget}")

    all_budgets = uow.budgets.get_all()
    print(f"[get_all] Total budgets: {len(all_budgets)}")

In [ ]:
# --- add_budget_to_user (operació de domini) ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    users = uow.users.get_all()
    user_id = users[0].id_user

    new_budget = Budget(
        id_user=user_id,
        month=6,
        year=2026,
        description="Pressupost Juny via operació de domini",
        total_limit=Decimal("2000.00")
    )
    uow.users.add_budget_to_user(user_id, new_budget)
    uow.commit()
    print(f"[add_budget_to_user] Budget id={new_budget.id_budget} afegit a usuari id={user_id}")

---
### Bloc 5 – `TransactionRepository`: add, get_paginated, get_by_category, add_tag_to_transaction

In [ ]:
# --- Preparació: afegir 15 transaccions per provar la paginació ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    budgets = uow.budgets.get_all()
    cats = uow.categories.get_all()
    methods = uow.payment_methods.get_all()
    types = uow.transaction_types.get_all()

    budget_id = budgets[0].id_budget
    cat_id = cats[0].id_category
    method_id = methods[0].id_method
    type_id = types[0].id_type

    for i in range(1, 16):
        t = Transaction(
            id_budget=budget_id,
            id_category=cat_id,
            id_method=method_id,
            id_type=type_id,
            amount=Decimal(str(i * 10)),
            date=date(2026, 5, i),
            description=f"Transacció de prova {i}",
            is_recurring=False
        )
        uow.transactions.add(t)

    uow.commit()
    total = uow.transactions.get_all()
    print(f"[add] Afegides 15 transaccions. Total ara: {len(total)}")

In [ ]:
# --- get_paginated() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    page1 = uow.transactions.get_paginated(page=1, page_size=5)
    page2 = uow.transactions.get_paginated(page=2, page_size=5)
    page3 = uow.transactions.get_paginated(page=3, page_size=5)

    print(f"[get_paginated] Pàgina 1: {len(page1)} transaccions  ids={[t.id_transaction for t in page1]}")
    print(f"[get_paginated] Pàgina 2: {len(page2)} transaccions  ids={[t.id_transaction for t in page2]}")
    print(f"[get_paginated] Pàgina 3: {len(page3)} transaccions  ids={[t.id_transaction for t in page3]}")

In [ ]:
# --- get_by_category() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    cats = uow.categories.get_all()
    cat_id = cats[0].id_category
    result = uow.transactions.get_by_category(cat_id)
    print(f"[get_by_category] Categoria id={cat_id}: {len(result)} transaccions")

In [ ]:
# --- add_tag_to_transaction (operació de domini) ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    # Creem un tag
    tag = Tag(name="Prova-Tag", color_code="#FF5733")
    uow.tags.add(tag)
    uow.commit()

    # Obtenim la primera transacció
    transactions = uow.transactions.get_all()
    tid = transactions[0].id_transaction

    # Recarreguem tag i transacció en la mateixa sessió
    fresh_tag = uow.tags.get(tag.id_tag)
    uow.transactions.add_tag_to_transaction(tid, fresh_tag)
    uow.commit()

    # Verifiquem
    tx = uow.transactions.get(tid)
    print(f"[add_tag_to_transaction] Transacció id={tid} té {len(tx.tags)} tag(s):")
    for tg in tx.tags:
        print(f"  → '{tg.name}' ({tg.color_code})")

---
### Bloc 6 – `BudgetCategoryRepository`: add, get (clau composta), get_by_budget, delete

In [ ]:
# --- add() + get() per clau composta ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    budgets = uow.budgets.get_all()
    cats = uow.categories.get_all()

    bc = BudgetCategory(
        id_budget=budgets[0].id_budget,
        id_category=cats[0].id_category,
        max_amount=Decimal("300.00"),
        alert_threshold=Decimal("0.80")
    )
    uow.budget_categories.add(bc)
    uow.commit()
    print(f"[add] BudgetCategory creada: budget={bc.id_budget}, cat={bc.id_category}")

    # get() per clau composta
    found = uow.budget_categories.get((bc.id_budget, bc.id_category))
    print(f"[get] Trobat: max_amount={found.max_amount}  alert={found.alert_threshold}")

In [ ]:
# --- get_by_budget() + delete() ---
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    budgets = uow.budgets.get_all()
    budget_id = budgets[0].id_budget

    bcs = uow.budget_categories.get_by_budget(budget_id)
    print(f"[get_by_budget] Budget id={budget_id}: {len(bcs)} categories")

    # delete() el primer
    to_delete = uow.budget_categories.get((bcs[0].id_budget, bcs[0].id_category))
    uow.budget_categories.delete(to_delete)
    uow.commit()
    print(f"[delete] BudgetCategory eliminada. Queden: {len(uow.budget_categories.get_by_budget(budget_id))}")

---
### Bloc 7 – `delete()` de User (amb rollback si cal)

In [ ]:
# Creem un usuari temporal i el borrem
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    temp_user = User(
        name="Temporal",
        surname="AEsborrar",
        email="delete.me@walletly.com",
        password="temp"
    )
    uow.users.add(temp_user)
    uow.commit()
    uid = temp_user.id_user
    print(f"[add] Usuari temporal creat: id={uid}")

with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    user_to_del = uow.users.get(uid)
    uow.users.delete(user_to_del)
    uow.commit()
    gone = uow.users.get(uid)
    print(f"[delete] Usuari id={uid} eliminat. Existeix? {gone is not None}")

In [ ]:
with SqlAlchemyUnitOfWork(SessionFactory) as uow:
    print("=" * 50)
    print("RESUM FINAL DE L'ESTAT DE LA BD")
    print("=" * 50)
    print(f"  Usuaris:           {len(uow.users.get_all())}")
    print(f"  Categories:        {len(uow.categories.get_all())}")
    print(f"  Mètodes pagament:  {len(uow.payment_methods.get_all())}")
    print(f"  Tipus transacció:  {len(uow.transaction_types.get_all())}")
    print(f"  Pressupostos:      {len(uow.budgets.get_all())}")
    print(f"  Transaccions:      {len(uow.transactions.get_all())}")
    print(f"  Tags:              {len(uow.tags.get_all())}")
    print("=" * 50)
    print("Tots els mètodes executats sense errors.")